# 07 — Двухэтапное обучение ансамбля

## Стратегия

```
┌────────────────────────────────────────────────────────────────────┐
│  ЭТАП 1: PRETRAIN  (~200-300k примеров, все источники)             │
│                                                                    │
│  ru_go_emotions + cedr + ru_izard + dusha + brighter_hf           │
│  + aniemore/resd + rureviews + rusentitweet                       │
│                                                                    │
│  loss=focal, class_weights=True, 3 эпохи                          │
└───────────────────────────┬────────────────────────────────────────┘
                            │  (инициализация весов)
                            ▼
┌────────────────────────────────────────────────────────────────────┐
│  ЭТАП 2: FINE-TUNE  (~15-20k примеров, нативный RU)               │
│                                                                    │
│  cedr + brighter_hf + aniemore/resd                               │
│  (полное покрытие всех 7 классов, высокое качество разметки)      │
│                                                                    │
│  loss=ce, lr×0.25, 3 эпохи                                        │
└────────────────────────────────────────────────────────────────────┘
```

Три модели обучаются по этой схеме, затем объединяются в ансамбль (ноутбук 05).

In [ ]:
import sys, os, json

PROJECT_ROOT = '/kaggle/input/sentiment-analysis' if os.path.exists('/kaggle') else os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

WORKING_DIR = '/kaggle/working' if os.path.exists('/kaggle') else '../results'
os.makedirs(WORKING_DIR, exist_ok=True)

import torch
import warnings
warnings.filterwarnings('ignore')

print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
if torch.cuda.is_available():
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1. Конфигурация

In [ ]:
from src.data_loader import EKMAN_LABEL_NAMES

LABEL_NAMES = EKMAN_LABEL_NAMES  # ['anger','disgust','fear','joy','sadness','surprise','neutral']
NUM_LABELS  = len(LABEL_NAMES)

# ── Три модели для ансамбля ────────────────────────────────────────
MODELS = {
    'rubert': {
        'model_name': 'blanchefort/rubert-base-cased-sentiment',
        'stage1_dir': f'{WORKING_DIR}/models/rubert_s1',
        'stage2_dir': f'{WORKING_DIR}/models/rubert',   # финальная модель
        's1_batch_size': 32,
        's2_batch_size': 32,
        's1_gradient_accumulation_steps': 1,
        's2_gradient_accumulation_steps': 1,
    },
    'xlmroberta': {
        'model_name': 'xlm-roberta-base',
        'stage1_dir': f'{WORKING_DIR}/models/xlmroberta_s1',
        'stage2_dir': f'{WORKING_DIR}/models/xlmroberta',
        's1_batch_size': 16,
        's2_batch_size': 16,
        's1_gradient_accumulation_steps': 2,  # эффективный batch=32
        's2_gradient_accumulation_steps': 2,
    },
    'rubert_tiny': {
        'model_name': 'cointegrated/rubert-tiny2',
        'stage1_dir': f'{WORKING_DIR}/models/rubert_tiny_s1',
        'stage2_dir': f'{WORKING_DIR}/models/rubert_tiny',
        's1_batch_size': 64,
        's2_batch_size': 64,
        's1_gradient_accumulation_steps': 1,
        's2_gradient_accumulation_steps': 1,
    },
}

# ── Гиперпараметры ─────────────────────────────────────────────────
# Stage 1
S1_EPOCHS     = 3
S1_LR         = 2e-5
S1_LOSS       = 'focal'
S1_GAMMA      = 2.0

# Stage 2
S2_EPOCHS     = 3
S2_LR         = 5e-6      # lr × 0.25 — осторожный fine-tune
S2_LOSS       = 'ce'
S2_SMOOTHING  = 0.05

MAX_LENGTH = 128
FP16       = True
SEED       = 42

# Какие модели обучать (False = пропустить, если уже обучена)
TRAIN_FLAGS = {
    'rubert':     True,
    'xlmroberta': True,
    'rubert_tiny': True,
}
print('Конфигурация загружена')

## 2. Загрузка датасетов

### 2а. Этап 1 — большой смешанный корпус

In [ ]:
from datasets import load_from_disk
from src.data_loader import (
    load_ru_go_emotions, load_cedr, load_ru_izard_emotions,
    load_brighter_hf, load_dusha, load_aniemore_resd,
    load_rureviews, load_rusentitweet,
    merge_datasets,
)
from src.preprocessor import preprocess_batch

S1_DATA_PATH = f'{WORKING_DIR}/stage1_data'

if os.path.exists(S1_DATA_PATH):
    print('Загружаем готовый Stage-1 датасет с диска...')
    stage1_ds = load_from_disk(S1_DATA_PATH)
else:
    print('Собираем Stage-1 датасет из всех источников...')
    s1_sources = {}

    # ru_go_emotions в режиме 'raw' (~211k) даёт ~4× больше fear и disgust чем 'simplified'
    for name, loader, kwargs in [
        ('ru_go_emotions_raw',  load_ru_go_emotions,    {'config': 'raw'}),
        ('cedr',                load_cedr,               {}),
        ('ru_izard',            load_ru_izard_emotions,  {}),
    ]:
        try:
            s1_sources[name] = loader(**kwargs)
        except Exception as e:
            print(f'  WARNING: {name} failed: {e}')
            # fallback: simplified GoEmotions
            if name == 'ru_go_emotions_raw':
                try:
                    s1_sources['ru_go_emotions'] = load_ru_go_emotions(config='simplified')
                except Exception as e2:
                    print(f'  WARNING: ru_go_emotions simplified also failed: {e2}')

    for name, loader in [
        ('brighter_hf',  load_brighter_hf),
        ('aniemore',     load_aniemore_resd),
        ('rureviews',    load_rureviews),
        ('rusentitweet', load_rusentitweet),
    ]:
        try:
            s1_sources[name] = loader()
        except Exception as e:
            print(f'  WARNING: {name} failed: {e}')

    # Dusha: anger/sadness/neutral/joy из ~300k транскриптов речи
    # max_samples на класс — не больше 10k angry, чтобы не перевесить
    try:
        s1_sources['dusha'] = load_dusha(max_samples=40_000)
    except Exception as e:
        print(f'  WARNING: dusha failed: {e}')

    stage1_raw = merge_datasets(s1_sources, test_size=0.15, val_size=0.15, seed=SEED)

    # Балансировка: не более MAX_PER_CLASS на класс в train
    import pandas as pd
    from datasets import Dataset, DatasetDict
    from sklearn.utils import resample
    from src.data_loader import EKMAN_ID2LABEL

    MAX_PER_CLASS = 15_000
    train_df = stage1_raw['train'].to_pandas()

    print('\nРаспределение ДО балансировки:')
    for lid in range(NUM_LABELS):
        cnt = (train_df['label'] == lid).sum()
        print(f'  {EKMAN_ID2LABEL[lid]:<12}: {cnt:>7,}')

    parts = []
    for lid in range(NUM_LABELS):
        sub = train_df[train_df['label'] == lid]
        if len(sub) == 0:
            print(f'  WARNING: нет примеров для {EKMAN_ID2LABEL[lid]}')
            continue
        if len(sub) > MAX_PER_CLASS:
            sub = resample(sub, n_samples=MAX_PER_CLASS, random_state=SEED, replace=False)
        parts.append(sub)
    train_df = pd.concat(parts).sample(frac=1, random_state=SEED).reset_index(drop=True)

    stage1_ds = DatasetDict({
        'train':      Dataset.from_pandas(train_df),
        'validation': stage1_raw['validation'],
        'test':       stage1_raw['test'],
    })

    # Очистка текстов
    stage1_ds = stage1_ds.map(
        lambda b: preprocess_batch(b, clean=True), batched=True, batch_size=1000
    )
    stage1_ds.save_to_disk(S1_DATA_PATH)
    print(f'Stage-1 датасет сохранён: {S1_DATA_PATH}')

print('\nStage-1 splits:')
for split in stage1_ds:
    print(f'  {split:12s}: {len(stage1_ds[split]):,}')

# Распределение классов в train
import pandas as pd
from src.data_loader import EKMAN_ID2LABEL
train_df = stage1_ds['train'].to_pandas()
print('\nРаспределение классов в Stage-1 train:')
total = len(train_df)
for lid in range(NUM_LABELS):
    cnt = (train_df['label'] == lid).sum()
    pct = cnt / total * 100
    bar = '█' * int(cnt / total * 50)
    print(f'  {EKMAN_ID2LABEL[lid]:<12}: {cnt:>7,}  ({pct:5.1f}%)  {bar}')


### 2б. Этап 2 — чистый нативный корпус

In [ ]:
from src.data_loader import load_stage2_clean

S2_DATA_PATH = f'{WORKING_DIR}/stage2_data'

if os.path.exists(S2_DATA_PATH):
    print('Загружаем готовый Stage-2 датасет с диска...')
    stage2_ds = load_from_disk(S2_DATA_PATH)
else:
    print('Собираем Stage-2 (чистый) датасет...')
    stage2_ds = load_stage2_clean(
        use_cedr=True,
        use_brighter_hf=True,
        use_aniemore=True,
        seed=SEED,
    )
    stage2_ds = stage2_ds.map(
        lambda b: preprocess_batch(b, clean=True), batched=True, batch_size=500
    )
    stage2_ds.save_to_disk(S2_DATA_PATH)
    print(f'Stage-2 датасет сохранён: {S2_DATA_PATH}')

from src.data_loader import EKMAN_ID2LABEL
print('\nStage-2 splits:')
for split in stage2_ds:
    print(f'  {split:12s}: {len(stage2_ds[split]):,}')

print('\nПокрытие классов в Stage-2 train:')
import pandas as pd
df2 = stage2_ds['train'].to_pandas()
counts = df2['label'].value_counts().sort_index()
for lid, cnt in counts.items():
    print(f'  {EKMAN_ID2LABEL[lid]:<12s}: {cnt:>5,}  ({cnt/len(df2)*100:.1f}%)')

## 3. Двухэтапное обучение ансамбля

Каждая из трёх моделей обучается по схеме pretrain → fine-tune.
Итоговые модели (stage2_dir) используются в ноутбуке 05 для ансамблирования.

In [ ]:
from src.trainer import train_two_stage

all_results = {}

for model_key, cfg in MODELS.items():
    if not TRAIN_FLAGS.get(model_key, False):
        print(f'\nПропускаем {model_key} (TRAIN_FLAGS={False})')
        continue

    print(f'\n{"#"*60}')
    print(f'  Модель: {cfg["model_name"]}')
    print(f'  Stage1: {cfg["stage1_dir"]}')
    print(f'  Stage2: {cfg["stage2_dir"]}')
    print('#'*60)

    os.makedirs(cfg['stage1_dir'], exist_ok=True)
    os.makedirs(cfg['stage2_dir'], exist_ok=True)

    model, tokenizer, r1, r2 = train_two_stage(
        model_name=cfg['model_name'],
        stage1_dataset=stage1_ds,
        stage2_dataset=stage2_ds,
        stage1_dir=cfg['stage1_dir'],
        stage2_dir=cfg['stage2_dir'],
        num_labels=NUM_LABELS,
        label_names=LABEL_NAMES,
        # Stage 1
        s1_epochs=S1_EPOCHS,
        s1_batch_size=cfg['s1_batch_size'],
        s1_lr=S1_LR,
        s1_loss_type=S1_LOSS,
        s1_focal_gamma=S1_GAMMA,
        s1_use_class_weights=True,
        s1_gradient_accumulation_steps=cfg['s1_gradient_accumulation_steps'],
        # Stage 2
        s2_epochs=S2_EPOCHS,
        s2_batch_size=cfg['s2_batch_size'],
        s2_lr=S2_LR,
        s2_loss_type=S2_LOSS,
        s2_label_smoothing=S2_SMOOTHING,
        s2_gradient_accumulation_steps=cfg['s2_gradient_accumulation_steps'],
        # Common
        max_length=MAX_LENGTH,
        fp16=FP16,
        seed=SEED,
    )

    all_results[model_key] = {'stage1': r1, 'stage2': r2}
    del model  # освобождаем VRAM перед следующей моделью
    import gc; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print('\n✓ Все модели обучены.')

## 4. Сравнение этапов

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.evaluation import evaluate_predictions, compare_models

rows = []
for model_key, cfg in MODELS.items():
    for stage, stage_dir in [('Stage 1', cfg['stage1_dir']), ('Stage 2 (final)', cfg['stage2_dir'])]:
        preds_path = os.path.join(stage_dir, 'test_preds.npy')
        if not os.path.exists(preds_path):
            continue
        preds  = np.load(os.path.join(stage_dir, 'test_preds.npy'))
        labels = np.load(os.path.join(stage_dir, 'test_labels.npy'))
        probs  = np.load(os.path.join(stage_dir, 'test_probs.npy'))
        m = evaluate_predictions(labels, preds, probs,
                                 model_name=f'{model_key} — {stage}',
                                 label_names=LABEL_NAMES)
        rows.append(m)

if rows:
    df = pd.DataFrame(rows).set_index('model')
    print('=== Stage 1 vs Stage 2 ===')
    print(df[['accuracy', 'f1_macro', 'f1_weighted']].round(4).to_string())

    compare_models(rows, save_path=f'{WORKING_DIR}/two_stage_comparison.png')

In [ ]:
# Per-class F1 после Stage 2 для каждой модели
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
if not axes.shape:
    axes = [axes]

for ax, (model_key, cfg) in zip(axes, MODELS.items()):
    stage2_dir = cfg['stage2_dir']
    if not os.path.exists(os.path.join(stage2_dir, 'test_preds.npy')):
        ax.set_title(f'{model_key}\n(не обучена)')
        continue

    preds  = np.load(os.path.join(stage2_dir, 'test_preds.npy'))
    labels = np.load(os.path.join(stage2_dir, 'test_labels.npy'))
    probs  = np.load(os.path.join(stage2_dir, 'test_probs.npy'))
    m = evaluate_predictions(labels, preds, probs, model_name=model_key, label_names=LABEL_NAMES)

    f1_per_class = [m.get(f'f1_{e}', 0) for e in LABEL_NAMES]
    colors = ['#e74c3c','#8e44ad','#2c3e50','#f39c12','#3498db','#1abc9c','#95a5a6']
    ax.barh(LABEL_NAMES, f1_per_class, color=colors)
    ax.set_xlim(0, 1)
    ax.set_title(f'{model_key}\nF1-macro = {m["f1_macro"]:.4f}')
    for i, v in enumerate(f1_per_class):
        ax.text(v + 0.01, i, f'{v:.2f}', va='center', fontsize=9)

axes[0].set_xlabel('F1-score')
plt.suptitle('Per-class F1 после двухэтапного обучения', fontsize=13)
plt.tight_layout()
plt.savefig(f'{WORKING_DIR}/per_class_f1_two_stage.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Сохранение конфигурации для ансамблирования

Создаём `ensemble_config.json` — ноутбук 05 считает его и знает пути к финальным моделям.

In [ ]:
ensemble_config = {
    'model_dirs': {
        k: v['stage2_dir'] for k, v in MODELS.items()
        if os.path.exists(os.path.join(v['stage2_dir'], 'config.json'))
    },
    'label_names': LABEL_NAMES,
    'training_strategy': 'two_stage',
    'stage1': {
        'epochs': S1_EPOCHS, 'lr': S1_LR,
        'loss': S1_LOSS, 'gamma': S1_GAMMA,
    },
    'stage2': {
        'epochs': S2_EPOCHS, 'lr': S2_LR,
        'loss': S2_LOSS, 'label_smoothing': S2_SMOOTHING,
        'corpus': 'cedr + brighter_hf + aniemore/resd',
    },
}

with open(f'{WORKING_DIR}/ensemble_config.json', 'w', encoding='utf-8') as f:
    json.dump(ensemble_config, f, ensure_ascii=False, indent=2)

with open(f'{WORKING_DIR}/label_names.json', 'w') as f:
    json.dump(LABEL_NAMES, f)

print('Конфигурация сохранена:')
print(json.dumps(ensemble_config, ensure_ascii=False, indent=2))

---
## Дальнейшие шаги

1. **Ноутбук 05** — ансамблирование трёх финальных моделей (soft voting, stacking)
2. **Ноутбук 06** — прикладной анализ (эмоциональные дуги, временные ряды)
3. **app/app.py** — Gradio-демо с ансамблем

```python
# Быстрый инференс на новом тексте:
from src.inference import EmotionClassifier
clf = EmotionClassifier([cfg['stage2_dir'] for cfg in MODELS.values()])
clf.predict('Мне очень страшно идти туда одному')
```